## 1. Zadatak

> 1. Zadatak: Ručno preuzmite pdf dokument "[Matching the Blanks: Distributional Similarity for Relation Learning](https://arxiv.org/pdf/1906.03158)" te pomoću prikladnih alata (npr. [pdfminer.six](https://pdfminersix.readthedocs.io/en/latest/)) izlučite sve naslove/podnaslove uključujući naslov rada.

In [1]:
from io import StringIO
from pdfminer.high_level import extract_text_to_fp
from pdfminer.layout import LAParams

output_string = StringIO()

with open('1906.03158v1.pdf', 'rb') as fin:
    extract_text_to_fp(fin, output_string, laparams=LAParams(), output_type='html', codec=None)

In [2]:
from bs4 import BeautifulSoup

html_string = output_string.getvalue()
soup = BeautifulSoup(html_string)

In [15]:
with open("1906.03158v1.html", "w") as f:
    f.write(soup.prettify())

Potražimo sve elemente tipa `<span>` koji imaju određeni stil teksta:

In [16]:
import re

regular_expression = r"font-family: NimbusRomNo9L-Medi; font-size:1[014]px"
result = [r for r in soup.find_all("span", style=re.compile(regular_expression))]

In [17]:
[r.get_text().strip() for r in result]

['Matching the Blanks: Distributional Similarity for Relation Learning',
 'Livio Baldini Soares',
 'Nicholas FitzGerald',
 'Jeffrey Ling',
 'Tom Kwiatkowski',
 'Abstract',
 '1',
 'Introduction',
 '2 Overview',
 'Task deﬁnition',
 'Contributions',
 '3 Architectures for Relation Learning',
 '3.1 Relation Classiﬁcation and Extraction',
 'Tasks',
 '3.2 Relation Representations from Deep',
 'Transformers Model',
 '3.2.1 Entity span identiﬁcation',
 'Standard input',
 'Positional embeddings',
 'Entity marker tokens',
 '3.3 Fixed length relation representation',
 '[',
 '] token',
 'Entity mention pooling',
 'Entity start state',
 '4.1 Learning Setup',
 '4 Learning by Matching the Blanks',
 '4.2',
 'Introducing Blanks',
 '4.3 Matching the Blanks Training',
 '5 Experimental Evaluation',
 '5.1 Few-shot Relation Matching',
 '5.2 Supervised Relation Extraction',
 '6 Conclusion and Future Work',
 'References']

U rezultatu, zbog raspona veličina teksta, imamo i imena autora te naglašene dijelove teksta. MOguća riješenja su:
1. Autori - Potražiti pojavljivanje naslova "Abstract" koji je uobičajeno prvi naslov te ignorirati rezultate prije istog.
2. Naglašeni dijelovi teksta - U ovom radu naslovi su numerirani, iz rezultata možemo izdvojiti one koji počinju znamenkom.

1:

In [ ]:
start = 0
for i in range(len(result)):
    if re.search(r"abstract", result[i].get_text(), re.IGNORECASE):
        start = i
        break

result = [result[0]] + result[start:] # Zadržavamo i prvi element koji je uobičajeno naslov rada.

In [20]:
[r.get_text().strip() for r in result]

['Matching the Blanks: Distributional Similarity for Relation Learning',
 'Abstract',
 '1',
 'Introduction',
 '2 Overview',
 'Task deﬁnition',
 'Contributions',
 '3 Architectures for Relation Learning',
 '3.1 Relation Classiﬁcation and Extraction',
 'Tasks',
 '3.2 Relation Representations from Deep',
 'Transformers Model',
 '3.2.1 Entity span identiﬁcation',
 'Standard input',
 'Positional embeddings',
 'Entity marker tokens',
 '3.3 Fixed length relation representation',
 '[',
 '] token',
 'Entity mention pooling',
 'Entity start state',
 '4.1 Learning Setup',
 '4 Learning by Matching the Blanks',
 '4.2',
 'Introducing Blanks',
 '4.3 Matching the Blanks Training',
 '5 Experimental Evaluation',
 '5.1 Few-shot Relation Matching',
 '5.2 Supervised Relation Extraction',
 '6 Conclusion and Future Work',
 'References']

2:

In [24]:
for i, r in enumerate(result):
    temp_txt = r.get_text().strip()
    hot_fix = re.search(r"(Abstract|Introduction|References)", temp_txt)
    if (i == 0 or re.match(r"\d", temp_txt) or hot_fix) and len(temp_txt) > 3:
        print(temp_txt)

Matching the Blanks: Distributional Similarity for Relation Learning
Abstract
Introduction
2 Overview
3 Architectures for Relation Learning
3.1 Relation Classiﬁcation and Extraction
3.2 Relation Representations from Deep
3.2.1 Entity span identiﬁcation
3.3 Fixed length relation representation
4.1 Learning Setup
4 Learning by Matching the Blanks
4.3 Matching the Blanks Training
5 Experimental Evaluation
5.1 Few-shot Relation Matching
5.2 Supervised Relation Extraction
6 Conclusion and Future Work
References


for r in result:

## 2. Zadatak

> 2. Zadatak: Izlučite sav tekstualni sadržaj istraživačkog rada (preuzetog dokumenta), zanemarite matematičke formule te informacije o radu (datum objave, imena autora, afilijacije, ...). ~~isključujući matematičke formule te meta-podatke.~~

Primjer sadržaja:
```html
<span style="font-family: NimbusRomNo9L-Regu; font-size:10px">
    Reading text to identify and extract relations be-
    <br/>
    tween entities has been a long standing goal in
    <br/>
   </span>
```

In [25]:
regular_expression = r"font-family: NimbusRomNo9L-Regu; font-size:10px"
result = [r for r in soup.find_all("span", style=re.compile(regular_expression))]

In [31]:
content = " ".join([r.get_text() for r in result])

In [33]:
print(content[:100])

Reading text to identify and extract relations be-
tween entities has been a long standing goal in
 


Naš sadržaj je oblikovan stupcima te je kao takav razlomljen. Tekst nam je potreban u formatu koji je moguće lakše pretraživati.

In [39]:
content_breakfree = re.sub(r"-\n","", content) # Tekst bez lomljnenja teksta 
content_breakfree_nonewlines = re.sub(r"\s+", " ", content_breakfree) # Tekst bez prelaska u novu liniju


In [47]:
print(content_breakfree_nonewlines[:100], "...")
print("Duljina teksta: ")
print("\t", len(content_breakfree_nonewlines), "znaka")
print("\t", len(content_breakfree_nonewlines.split()), "riječi")


Reading text to identify and extract relations between entities has been a long standing goal in nat ...
Duljina teksta: 
	 21556 znaka
	 3548 riječi


## Zadatak 3

> Zadatak 3: Izlučite sav tekstualni sadržaj iz pdf dokumenta. Izdvojite tekst pod naslovom "Abstract" i "References" u zasebne varijable.

In [ ]:
# Sadržaj: Preuzeto iz zadatka 2
regular_expression = r"font-family: NimbusRomNo9L-Regu; font-size:10px"
result = [r for r in soup.find_all("span", style=re.compile(regular_expression))]

content = " ".join([r.get_text() for r in result])

content_breakfree = re.sub(r"-\n","", content) # Tekst bez lomljnenja teksta 
content_breakfree_nonewlines = re.sub(r"\s+", " ", content_breakfree) # Tekst bez prelaska u novu liniju
content = content_breakfree_nonewlines

In [49]:
# Abstract
regular_expression = r"font-family: NimbusRomNo9L-Regu; font-size:9px"
result = [r for r in soup.find_all("span", style=re.compile(regular_expression))]

abstract = " ".join([r.get_text() for r in result])
abstract_breakfree = re.sub(r"-\n","", abstract) # Tekst bez lomljnenja teksta 
abstract_breakfree_nonewlines = re.sub(r"\s+", " ", abstract_breakfree) # Tekst bez prelaska u novu liniju
abstract = abstract_breakfree_nonewlines


In [50]:
abstract

'General purpose relation extractors, which can model arbitrary relations, are a core aspiration in information extraction. Efforts have been made to build general purpose extractors that represent relations with their surface forms, or which jointly embed surface forms with rela tions from an existing knowledge graph. How ever, both of these approaches are limited in their ability to generalize. In this paper, we build on extensions of Harris’ distributional hypothesis to relations, as well as recent ad vances in learning text representations (specif ically, BERT), to build task agnostic relation representations solely from entity-linked text. We show that these representations signiﬁcantly outperform previous work on exemplar based relation extraction (FewRel) even with out using any of that task’s training data. We also show that models initialized with our task agnostic representations, and then tuned on su pervised relation extraction datasets, signiﬁcantly outperform the previous